In [1]:
%pip install -q customtkinter pillow numpy scikit-image matplotlib cryptography

Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [2]:
import os
import io
import csv
import time
import hashlib

import numpy as np
from PIL import Image, ImageOps
from skimage.metrics import structural_similarity         
from cryptography.hazmat.primitives.ciphers.aead import AESGCM
from cryptography.exceptions import InvalidTag

In [3]:
MAGIC = b"LSB1"
HEADER_SIZE = 9


def load_image(path):
    """Open an image file and return it as an RGB picture."""
    img = Image.open(path)
    img = ImageOps.exif_transpose(img)     
    return img.convert("RGB")


def capacity_bytes(cover_array):
    """How many bytes fit? We use 1 bit from each colour value (R, G and B of every pixel)."""
    return cover_array.size // 8 - HEADER_SIZE


def embed_data(cover_array, payload, encrypted=False):
    """Write the bytes in `payload` into the last bit of the cover's colour values."""
    if len(payload) > capacity_bytes(cover_array):
        raise ValueError(
            f"The document needs {len(payload):,} bytes but this cover image can only hold "
            f"{capacity_bytes(cover_array):,} bytes. Choose a larger cover image."
        )

    header = MAGIC + bytes([int(encrypted)]) + len(payload).to_bytes(4, "big")
    bits = np.unpackbits(np.frombuffer(header + payload, dtype=np.uint8))

    flat = cover_array.flatten()

    flat[:len(bits)] = (flat[:len(bits)] & 0xFE) | bits

    return flat.reshape(cover_array.shape)


def read_header(stego_array):
    """Peek at the header. Returns (is_encrypted, payload_length), or None if nothing is hidden."""
    flat = stego_array.reshape(-1)
    header = np.packbits(flat[:HEADER_SIZE * 8] & 1).tobytes()  
    if header[:4] != MAGIC:
        return None
    return header[4] == 1, int.from_bytes(header[5:9], "big")


def extract_data(stego_array):
    """EXTRACT: read the last bits back and rebuild the hidden bytes."""
    info = read_header(stego_array)
    if info is None:
        raise ValueError(
            "No hidden document found. Check that you chose the PNG stego image "
            "(JPG compression destroys hidden data)."
        )
    encrypted, length = info
    end = (HEADER_SIZE + length) * 8
    flat = stego_array.reshape(-1)
    if end > flat.size:
        raise ValueError("The hidden data is damaged (its length is larger than the image).")

    bits = flat[HEADER_SIZE * 8:end] & 1              
    return encrypted, np.packbits(bits).tobytes()     

In [4]:
def derive_key(password, salt):
    """Turn a password into a 256-bit AES key"""
    return hashlib.pbkdf2_hmac("sha256", password.encode("utf-8"), salt, 200_000, dklen=32)


def encrypt(data, password):
    """AES-256-GCM encryption. Output = salt (16) + nonce (12) + ciphertext and tag."""
    salt, nonce = os.urandom(16), os.urandom(12)
    return salt + nonce + AESGCM(derive_key(password, salt)).encrypt(nonce, data, None)


def decrypt(blob, password):
    """Reverse of encrypt(). Raises InvalidTag if the password is wrong or the data was changed."""
    salt, nonce, ciphertext = blob[:16], blob[16:28], blob[28:]
    return AESGCM(derive_key(password, salt)).decrypt(nonce, ciphertext, None)


def file_sha256(path):
    """Fingerprint of a file. Same file = same fingerprint, so we can prove nothing changed."""
    with open(path, "rb") as f:
        return hashlib.sha256(f.read()).hexdigest()


def pack_document(path):
    """Read the legal document and wrap it:  [SHA-256 (32 bytes) | extension length | extension | file bytes]"""
    with open(path, "rb") as f:
        raw = f.read()
    ext = os.path.splitext(path)[1].lower().encode()          
    return hashlib.sha256(raw).digest() + bytes([len(ext)]) + ext + raw


def unpack_document(package):
    """Reverse of pack_document(). Returns (file bytes, extension, integrity_ok)."""
    digest = package[:32]
    ext_len = package[32]
    ext = package[33:33 + ext_len].decode(errors="replace")
    raw = package[33 + ext_len:]
    return raw, ext, hashlib.sha256(raw).digest() == digest


def hide_document(cover_array, doc_path, password=""):
    """Full hiding pipeline: pack -> (encrypt) -> embed. Returns the stego image array."""
    package = pack_document(doc_path)
    if password:
        package = encrypt(package, password)
    return embed_data(cover_array, package, encrypted=bool(password))


def extract_document(stego_array, password=""):
    """Full extraction pipeline: extract -> (decrypt) -> unpack. Returns (bytes, extension, integrity_ok)."""
    encrypted, package = extract_data(stego_array)
    if encrypted:
        if not password:
            raise ValueError("This document is password protected. Enter the password and try again.")
        try:
            package = decrypt(package, password)
        except InvalidTag:
            raise ValueError("Wrong password, or the stego image was modified after hiding.")
    return unpack_document(package)

In [5]:
def psnr(original, modified):
    """Peak Signal-to-Noise Ratio in dB. Higher = more similar. Above ~40 dB the change is invisible."""
    diff = original.astype(np.int16) - modified.astype(np.int16)   
    mse = np.mean(diff.astype(np.float32) ** 2, dtype=np.float64)   
    if mse == 0:
        return float("inf")                                          
    return 10 * np.log10(255 ** 2 / mse)


def ssim(original, modified):
    """Structural Similarity Index. 1.0 = identical, closer to 1 = better."""
    return structural_similarity(original, modified, channel_axis=2, data_range=255)


def evaluate_size(cover_array, size_bytes):
    """Hide a test document of `size_bytes`, extract it again, and measure everything."""
    payload = os.urandom(size_bytes)

    t0 = time.perf_counter()
    stego = embed_data(cover_array, payload)
    embed_ms = (time.perf_counter() - t0) * 1000

    t0 = time.perf_counter()
    _, recovered = extract_data(stego)
    extract_ms = (time.perf_counter() - t0) * 1000

    return {
        "size_kb": size_bytes / 1024,
        "used_pct": 100 * (HEADER_SIZE + size_bytes) / (cover_array.size // 8),
        "embed_ms": embed_ms,
        "extract_ms": extract_ms,
        "psnr_db": psnr(cover_array, stego),
        "ssim": ssim(cover_array, stego),
        "exact_match": recovered == payload,
    }


def save_results_csv(rows, path):
    """Write the evaluation table to a CSV file (opens in Excel) for the lab report."""
    with open(path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        writer.writeheader()
        writer.writerows(rows)

In [6]:
test_cover = np.random.randint(0, 256, (200, 200, 3), dtype=np.uint8)
test_doc = os.urandom(1000)

test_stego = embed_data(test_cover, test_doc)
_, test_recovered = extract_data(test_stego)

print("Hidden and recovered correctly:", test_recovered == test_doc)
print("Capacity of a 200x200 image   :", capacity_bytes(test_cover), "bytes")
print("PSNR                          :", round(psnr(test_cover, test_stego), 2), "dB")
print("Wrong-password check          :", end=" ")
try:
    decrypt(encrypt(b"secret", "right"), "wrong")
except InvalidTag:
    print("rejected, as expected")

Hidden and recovered correctly: True
Capacity of a 200x200 image   : 14991 bytes
PSNR                          : 62.83 dB
Wrong-password check          : rejected, as expected


In [7]:
import customtkinter as ctk
from tkinter import filedialog, messagebox
from matplotlib.figure import Figure
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg

ctk.set_appearance_mode("dark")
ctk.set_default_color_theme("blue")

ACCENT, ACCENT_HOVER = "#2f7f79", "#266761"        
NEUTRAL, NEUTRAL_HOVER = "#3b4046", "#4a5057"      
LINE_TEAL, LINE_AMBER = "#4fb3a9", "#e9a23b"       
RED, GREEN = "#d9534f", "#5cb85c"
CHART_BG = "#242424"

IMAGE_TYPES = [("Image files", "*.jpg *.jpeg *.png *.bmp"), ("All files", "*.*")]
LOSSLESS_TYPES = [("PNG / BMP images", "*.png *.bmp"), ("All files", "*.*")]


class ImagePicker(ctk.CTkFrame):
    """A reusable box: title + 'Choose image' button + preview + file info.
    Leave filetypes empty to get a preview-only box (used for the recovered document)."""

    def __init__(self, master, title, filetypes=None, on_select=None):
        super().__init__(master, corner_radius=12)
        self.title_text = title
        self.filetypes = filetypes
        self.on_select = on_select      
        self.path = None                
        self.image = None               
        self.array = None               

        ctk.CTkLabel(self, text=title, font=ctk.CTkFont(size=15, weight="bold")).pack(pady=(10, 4))
        if filetypes:
            ctk.CTkButton(self, text="Choose image...", width=130, fg_color=NEUTRAL,
                          hover_color=NEUTRAL_HOVER, command=self.browse).pack(pady=4)
        self.preview = ctk.CTkLabel(self, text="No image selected", width=240, height=140, text_color="gray")
        self.preview.pack(padx=10, pady=4)
        self.info = ctk.CTkLabel(self, text="", text_color="gray", justify="center")
        self.info.pack(pady=(0, 10))

    def browse(self):
        path = filedialog.askopenfilename(title=self.title_text, filetypes=self.filetypes)
        if not path:
            return                      
        try:
            image = load_image(path)
        except Exception as e:
            messagebox.showerror("Cannot open image", f"This file could not be opened as an image.\n\n{e}")
            return
        self.path = path
        size_kb = os.path.getsize(path) / 1024
        self.show(image, f"{os.path.basename(path)}\n{image.width} x {image.height} px  |  {size_kb:,.1f} KB")
        if self.on_select:
            self.on_select()

    def show(self, image, info_text):
        self.image = image
        self.array = np.array(image)
        thumb = image.copy()
        thumb.thumbnail((240, 140))
        self._photo = ctk.CTkImage(light_image=thumb, dark_image=thumb, size=thumb.size)   
        self.preview.configure(image=self._photo, text="")
        self.info.configure(text=info_text)


class StegoApp(ctk.CTk):
    def __init__(self):
        super().__init__()
        self.title("Secure Legal Document Hiding - LSB Image Steganography")
        self.geometry("1000x720")
        self.minsize(900, 640)
        self.mono = ctk.CTkFont(family="Courier", size=12)
        self.eval_cover = None          
        self.eval_rows = []             

        ctk.CTkLabel(self, text="Secure Legal Document Hiding",
                     font=ctk.CTkFont(size=24, weight="bold")).pack(pady=(14, 0))
        ctk.CTkLabel(self, text="Hide a confidential document inside an ordinary image using LSB steganography",
                     text_color="gray").pack()

        tabs = ctk.CTkTabview(self)
        tabs.pack(fill="both", expand=True, padx=15, pady=(8, 15))
        self.build_hide_tab(tabs.add("Hide document"))
        self.build_extract_tab(tabs.add("Extract document"))
        self.build_eval_tab(tabs.add("Performance evaluation"))

    def log(self, box, text):
        """Replace the text in a result box and redraw the window right away."""
        box.delete("1.0", "end")
        box.insert("1.0", text)
        self.update_idletasks()

    def primary_button(self, master, text, command, **kwargs):
        return ctk.CTkButton(master, text=text, command=command, fg_color=ACCENT, hover_color=ACCENT_HOVER,
                             font=ctk.CTkFont(size=14, weight="bold"), height=38, **kwargs)


    def build_hide_tab(self, tab):
        tab.grid_columnconfigure((0, 1), weight=1)
        tab.grid_rowconfigure(4, weight=1)

        self.cover_box = ImagePicker(tab, "Cover image", IMAGE_TYPES, on_select=self.update_capacity_bar)
        self.cover_box.grid(row=0, column=0, padx=8, pady=8, sticky="nsew")
        self.doc_box = ImagePicker(tab, "Legal document to hide", IMAGE_TYPES, on_select=self.update_capacity_bar)
        self.doc_box.grid(row=0, column=1, padx=8, pady=8, sticky="nsew")

        self.cap_label = ctk.CTkLabel(tab, text="Choose a cover image and a legal document to see if it fits.",
                                      text_color="gray")
        self.cap_label.grid(row=1, column=0, columnspan=2, sticky="w", padx=12)
        self.cap_bar = ctk.CTkProgressBar(tab, progress_color=ACCENT)
        self.cap_bar.set(0)
        self.cap_bar.grid(row=2, column=0, columnspan=2, sticky="ew", padx=12, pady=(2, 8))

        row = ctk.CTkFrame(tab, fg_color="transparent")
        row.grid(row=3, column=0, columnspan=2, sticky="ew", padx=8)
        self.hide_pw = ctk.CTkEntry(row, width=300, height=38, show="*",
                                    placeholder_text="Password (optional, leave empty for no encryption)")
        self.hide_pw.pack(side="left", padx=4)
        self.show_pw = ctk.CTkCheckBox(row, text="Show", width=70, command=self.toggle_password)
        self.show_pw.pack(side="left", padx=4)
        self.primary_button(row, "Hide document", self.do_hide, width=170).pack(side="right", padx=4)

        self.hide_out = ctk.CTkTextbox(tab, height=150, font=self.mono)
        self.hide_out.grid(row=4, column=0, columnspan=2, sticky="nsew", padx=8, pady=8)

    def toggle_password(self):
        self.hide_pw.configure(show="" if self.show_pw.get() else "*")

    def update_capacity_bar(self):
        if self.cover_box.array is None:
            return
        capacity = capacity_bytes(self.cover_box.array)
        text = f"This cover image can hide up to {capacity / 1024:,.1f} KB."
        if self.doc_box.path:
            need = os.path.getsize(self.doc_box.path)
            ratio = need / capacity
            text += f"   Your document is {need / 1024:,.1f} KB, which uses {ratio * 100:.1f}% of the capacity."
            if ratio >= 1:
                text += "  It is too large: choose a bigger cover image."
            self.cap_bar.set(min(ratio, 1))
            self.cap_bar.configure(progress_color=RED if ratio >= 1 else ACCENT)
        self.cap_label.configure(text=text)

    def do_hide(self):
        if self.cover_box.array is None or not self.doc_box.path:
            messagebox.showwarning("Missing image", "Choose both a cover image and a legal document first.")
            return

        cover_name = os.path.splitext(os.path.basename(self.cover_box.path))[0]
        save_path = filedialog.asksaveasfilename(
            title="Save the stego image (PNG)", defaultextension=".png",
            initialfile=f"stego_{cover_name}.png", filetypes=[("PNG image", "*.png")])
        if not save_path:
            return
        save_path = os.path.splitext(save_path)[0] + ".png"       

        password = self.hide_pw.get()
        self.log(self.hide_out, "Working... large images can take a few seconds.")
        try:
            t0 = time.perf_counter()
            stego = hide_document(self.cover_box.array, self.doc_box.path, password)
            hide_time = time.perf_counter() - t0

            Image.fromarray(stego).save(save_path, format="PNG")
            quality_psnr = psnr(self.cover_box.array, stego)
            quality_ssim = ssim(self.cover_box.array, stego)
        except Exception as e:
            self.log(self.hide_out, f"Could not hide the document.\n\n{e}")
            return

        doc_kb = os.path.getsize(self.doc_box.path) / 1024
        capacity = capacity_bytes(self.cover_box.array)
        self.log(self.hide_out, "\n".join([
            "Document hidden successfully",
            "-" * 64,
            f"Cover image        : {os.path.basename(self.cover_box.path)}",
            f"Legal document     : {os.path.basename(self.doc_box.path)} ({doc_kb:,.1f} KB)",
            f"Protection         : {'AES-256-GCM password encryption' if password else 'none (no password set)'}",
            f"Hiding time        : {hide_time:.3f} s  (hash + encryption + embedding)",
            f"PSNR               : {quality_psnr:.2f} dB  (above 40 dB the change is invisible)",
            f"SSIM               : {quality_ssim:.6f}  (1.0 means identical)",
            f"Capacity used      : {os.path.getsize(self.doc_box.path) / capacity * 100:.1f} %",
            f"Document SHA-256   : {file_sha256(self.doc_box.path)}",
            f"Stego image saved  : {save_path}",
            "",
            "Share the PNG as it is. Converting it to JPG would destroy the hidden document.",
        ]))


    def build_extract_tab(self, tab):
        tab.grid_columnconfigure((0, 1), weight=1)
        tab.grid_rowconfigure(3, weight=1)

        self.stego_box = ImagePicker(tab, "Stego image", LOSSLESS_TYPES, on_select=self.check_stego)
        self.stego_box.grid(row=0, column=0, padx=8, pady=8, sticky="nsew")
        self.out_box = ImagePicker(tab, "Recovered document")        
        self.out_box.grid(row=0, column=1, padx=8, pady=8, sticky="nsew")

        self.stego_status = ctk.CTkLabel(tab, text="Choose the stego image to check for a hidden document.",
                                         text_color="gray")
        self.stego_status.grid(row=1, column=0, columnspan=2, sticky="w", padx=12)

        row = ctk.CTkFrame(tab, fg_color="transparent")
        row.grid(row=2, column=0, columnspan=2, sticky="ew", padx=8, pady=8)
        self.ext_pw = ctk.CTkEntry(row, width=300, height=38, show="*",
                                   placeholder_text="Password (only if one was used)")
        self.ext_pw.pack(side="left", padx=4)
        self.primary_button(row, "Extract document", self.do_extract, width=170).pack(side="right", padx=4)

        self.ext_out = ctk.CTkTextbox(tab, height=150, font=self.mono)
        self.ext_out.grid(row=3, column=0, columnspan=2, sticky="nsew", padx=8, pady=8)

    def check_stego(self):
        """Runs right after the user picks a stego image: look for the header."""
        info = read_header(self.stego_box.array)
        if info is None:
            self.stego_status.configure(text="No hidden document found in this image.", text_color=RED)
        else:
            encrypted, length = info
            kind = "password protected" if encrypted else "not encrypted"
            self.stego_status.configure(text=f"Hidden document found: {length / 1024:,.1f} KB, {kind}.",
                                        text_color=GREEN)

    def do_extract(self):
        if self.stego_box.array is None:
            messagebox.showwarning("Missing image", "Choose the stego image first.")
            return
        self.log(self.ext_out, "Working...")
        try:
            t0 = time.perf_counter()
            raw, ext, intact = extract_document(self.stego_box.array, self.ext_pw.get())
            extract_time = time.perf_counter() - t0
        except Exception as e:
            self.log(self.ext_out, f"Could not extract the document.\n\n{e}")
            return

        try:                                                      
            doc = Image.open(io.BytesIO(raw)).convert("RGB")
            self.out_box.show(doc, f"{len(raw) / 1024:,.1f} KB  |  {doc.width} x {doc.height} px")
        except Exception:
            self.out_box.info.configure(text="(no preview for this file type)")

        ext = ext or ".bin"
        save_path = filedialog.asksaveasfilename(
            title="Save the recovered document", defaultextension=ext,
            initialfile=f"recovered_document{ext}", filetypes=[(f"{ext} file", f"*{ext}"), ("All files", "*.*")])
        if save_path:
            with open(save_path, "wb") as f:
                f.write(raw)

        self.log(self.ext_out, "\n".join([
            "Document extracted" if intact else "Document extracted, but it is damaged",
            "-" * 64,
            f"Document size      : {len(raw) / 1024:,.1f} KB ({ext})",
            f"Extraction time    : {extract_time:.3f} s",
            "Integrity check    : " + ("passed, the SHA-256 matches the original" if intact
                                       else "failed, the hidden data was changed after hiding"),
            f"Recovered SHA-256  : {hashlib.sha256(raw).hexdigest()}",
            f"Saved to           : {save_path if save_path else 'not saved'}",
        ]))

    def build_eval_tab(self, tab):
        top = ctk.CTkFrame(tab, fg_color="transparent")
        top.pack(fill="x", padx=8, pady=(8, 2))
        ctk.CTkButton(top, text="Choose cover image...", fg_color=NEUTRAL, hover_color=NEUTRAL_HOVER,
                      command=self.pick_eval_cover).pack(side="left", padx=4)
        self.eval_info = ctk.CTkLabel(top, text="No cover image chosen", text_color="gray")
        self.eval_info.pack(side="left", padx=8)

        row = ctk.CTkFrame(tab, fg_color="transparent")
        row.pack(fill="x", padx=8, pady=4)
        ctk.CTkLabel(row, text="Document sizes to test (KB):").pack(side="left", padx=4)
        self.sizes_entry = ctk.CTkEntry(row, width=260)
        self.sizes_entry.insert(0, "10, 50, 100, 250, 500, 1000")
        self.sizes_entry.pack(side="left", padx=4)
        self.primary_button(row, "Run evaluation", self.run_eval, width=150).pack(side="left", padx=8)
        self.csv_btn = ctk.CTkButton(row, text="Save results as CSV", fg_color=NEUTRAL, hover_color=NEUTRAL_HOVER,
                                     state="disabled", command=self.save_csv)
        self.csv_btn.pack(side="left", padx=4)

        self.eval_out = ctk.CTkTextbox(tab, height=165, font=self.mono)
        self.eval_out.pack(fill="x", padx=12, pady=6)

        chart_frame = ctk.CTkFrame(tab, fg_color=CHART_BG)
        chart_frame.pack(fill="both", expand=True, padx=12, pady=(0, 8))
        self.fig = Figure(figsize=(9, 2.6), facecolor=CHART_BG)
        self.canvas = FigureCanvasTkAgg(self.fig, master=chart_frame)
        self.canvas.get_tk_widget().pack(fill="both", expand=True)

    def pick_eval_cover(self):
        path = filedialog.askopenfilename(title="Cover image for the evaluation", filetypes=IMAGE_TYPES)
        if not path:
            return
        try:
            self.eval_cover = np.array(load_image(path))
        except Exception as e:
            messagebox.showerror("Cannot open image", f"This file could not be opened as an image.\n\n{e}")
            return
        h, w, _ = self.eval_cover.shape
        self.eval_info.configure(
            text=f"{os.path.basename(path)}  |  {w} x {h} px  |  capacity {capacity_bytes(self.eval_cover) / 1024:,.1f} KB",
            text_color="white")

    def run_eval(self):
        if self.eval_cover is None:
            messagebox.showwarning("Missing image", "Choose a cover image first.")
            return
        try:
            sizes_kb = sorted(float(s) for s in self.sizes_entry.get().split(",") if s.strip())
            if not sizes_kb or sizes_kb[0] <= 0:
                raise ValueError
        except ValueError:
            messagebox.showerror("Invalid sizes", "Enter positive numbers separated by commas, for example: 10, 50, 100")
            return

        capacity = capacity_bytes(self.eval_cover)
        rows, skipped = [], []
        for kb in sizes_kb:
            if int(kb * 1024) > capacity:
                skipped.append(kb)
                continue
            self.log(self.eval_out, f"Testing {kb:g} KB ...  ({len(rows) + 1} of {len(sizes_kb) - len(skipped)})")
            rows.append(evaluate_size(self.eval_cover, int(kb * 1024)))

        if not rows:
            self.log(self.eval_out, "None of these sizes fit in the cover image. Use smaller sizes or a larger cover image.")
            return
        self.eval_rows = rows
        self.csv_btn.configure(state="normal")

        lines = [f"{'Size (KB)':>10}{'Used %':>9}{'Embed (ms)':>12}{'Extract (ms)':>14}{'PSNR (dB)':>11}{'SSIM':>10}{'Exact?':>8}",
                 "-" * 74]
        for r in rows:
            lines.append(f"{r['size_kb']:>10.1f}{r['used_pct']:>9.1f}{r['embed_ms']:>12.1f}{r['extract_ms']:>14.1f}"
                         f"{r['psnr_db']:>11.2f}{r['ssim']:>10.6f}{'yes' if r['exact_match'] else 'NO':>8}")
        lines.append("")
        lines.append(f"Hiding capacity of this cover image: {capacity / 1024:,.1f} KB ({capacity:,} bytes)")
        if skipped:
            lines.append("Skipped (too large for this cover): " + ", ".join(f"{k:g} KB" for k in skipped))
        self.log(self.eval_out, "\n".join(lines))
        self.draw_charts(rows)

    def draw_charts(self, rows):
        sizes = [r["size_kb"] for r in rows]
        psnr_values = [r["psnr_db"] if np.isfinite(r["psnr_db"]) else np.nan for r in rows]

        self.fig.clear()
        ax1 = self.fig.add_subplot(1, 2, 1)
        ax2 = self.fig.add_subplot(1, 2, 2)

        ax1.plot(sizes, [r["embed_ms"] for r in rows], marker="o", color=LINE_TEAL, label="Embedding")
        ax1.plot(sizes, [r["extract_ms"] for r in rows], marker="s", color=LINE_AMBER, label="Extraction")
        ax1.set_title("Time vs document size")
        ax1.set_ylabel("Time (ms)")
        ax1.legend(facecolor=CHART_BG, edgecolor="gray", labelcolor="white")

        ax2.plot(sizes, psnr_values, marker="o", color=LINE_TEAL, label="PSNR")
        ax2.axhline(40, color=RED, linestyle="--", linewidth=1, label="40 dB (good quality)")
        ax2.set_title("Image quality vs document size")
        ax2.set_ylabel("PSNR (dB)")
        ax2.legend(facecolor=CHART_BG, edgecolor="gray", labelcolor="white")

        for ax in (ax1, ax2):
            ax.set_xlabel("Document size (KB)")
            ax.set_facecolor(CHART_BG)
            ax.tick_params(colors="white")
            ax.title.set_color("white")
            ax.xaxis.label.set_color("white")
            ax.yaxis.label.set_color("white")
            ax.grid(alpha=0.2)
            for spine in ax.spines.values():
                spine.set_color("gray")

        self.fig.tight_layout()
        self.canvas.draw()

    def save_csv(self):
        path = filedialog.asksaveasfilename(title="Save evaluation results", defaultextension=".csv",
                                            initialfile="evaluation_results.csv", filetypes=[("CSV file", "*.csv")])
        if path:
            save_results_csv(self.eval_rows, path)
            messagebox.showinfo("Saved", f"Results saved to:\n{path}")

In [8]:
app = StegoApp()
app.mainloop()